In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# データ分割
from sklearn.model_selection import train_test_split

# 線形モデル
from sklearn.ensemble import RandomForestClassifier

# グラフをアウトプット行に出力するためのマジックコマンド
%matplotlib inline

# 精度評価
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from lazypredict.Supervised import LazyClassifier 
import lightgbm as lgb
import optuna

In [2]:
train = pd.read_csv('../../data/processed/train_card_tenure.csv')
test = pd.read_csv('../../data/processed/test_card_tenure.csv')

In [3]:
# 説明変数と目的変数に分割

# 説明変数
X = train.drop(['Exited'],axis=1)
# 目的変数
y = train['Exited']

In [4]:
# データの分割
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    random_state=42,
                                                    stratify=y)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (11250, 25), X_test: (3750, 25)


## 複数のモデルでROCを比較

In [5]:
clf = LazyClassifier(ignore_warnings=True, predictions=True)   #設定
models, predictions = clf.fit(X_train, X_test, y_train, y_test) #実行

In [6]:
models

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Precision,Recall,Time Taken
Model,,,,,,,
NearestCentroid,0.845867,0.841935,0.915990,0.854952,0.877764,0.845867,0.112499
BernoulliNB,0.867467,0.835573,0.914575,0.871987,0.880351,0.867467,0.101011
LinearDiscriminantAnalysis,0.895467,0.821565,0.927562,0.893508,0.892491,0.895467,0.117684
AdaBoostClassifier,0.900000,0.821010,0.929562,0.897239,0.896483,0.900000,0.890812
LGBMClassifier,0.894667,0.820577,0.929638,0.892723,0.891691,0.894667,0.318904
SGDClassifier,0.890667,0.820009,0.921897,0.889332,0.888402,0.890667,0.191880
LogisticRegression,0.898400,0.819033,0.929850,0.895655,0.894825,0.898400,0.112135
LinearSVC,0.899467,0.818245,0.929779,0.896452,0.895780,0.899467,0.126217
CalibratedClassifierCV,0.898667,0.817742,0.929812,0.895719,0.894977,0.898667,0.314254


## LightGBMを用いてモデルを作成する

In [7]:
dtrain = lgb.Dataset(X_train, y_train)
dvalid = lgb.Dataset(X_test, y_test)

params = {
    'objective': 'binary',
    'metrics': 'auc',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'learning_rate': 0.01   
}

model = lgb.train(params, dtrain, num_boost_round=1000)
pred_prob = model.predict(X_test)

## Optunaを用いてhyper parameterのチューニングを行う

In [8]:
# 目的関数の定義
def objective(trial):
    dtrain = lgb.Dataset(X_train, y_train)
    dvalid = lgb.Dataset(X_test, y_test)

    params = {
        'objective': 'binary',
        'metrics': 'auc',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'lambda_l1'         : trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2'         : trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),

        'force_col_wise':True,
        'random_state': 0,
    }

    model = lgb.train(
        params=params,
        train_set=dtrain,
        num_boost_round=100,
    )

    pred = model.predict(X_test, num_iteration=model.best_iteration)
    score = roc_auc_score(y_test, pred)
    
    return score

In [9]:
# 最適化の実行
optuna.logging.disable_default_handler()
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42), )
study.optimize(objective, n_trials=60)

In [10]:
# 最も精度の高いパラメータ
print('＝＝＝＝ベストパラメーター＝＝＝＝＝')
print(study.best_params)

# 最も精度の高いパラメータで学習を再度実行
best_params = {
        'objective': 'binary',
        'metrics': 'auc',
        'boosting_type': 'gbdt',
        'verbosity': -1
    }
best_params.update(study.best_params)

＝＝＝＝ベストパラメーター＝＝＝＝＝
{'lambda_l1': 2.9403833227233243, 'lambda_l2': 2.0805642015558648e-08, 'learning_rate': 0.05467636803348411, 'num_leaves': 17, 'feature_fraction': 0.7450442924057548, 'bagging_fraction': 0.6245570485884003, 'bagging_freq': 5, 'min_child_samples': 46}


## 層化K分割を用いてクロスバリデーションを行い、モデルの性能を評価する

In [11]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=43)
valid_score = []

for fold, (train_index, valid_index) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[train_index], X.iloc[valid_index]
    y_tr, y_va = y.iloc[train_index], y.iloc[valid_index]
    set_tr = lgb.Dataset(X_tr, y_tr)
    
    model_tmp = lgb.train(
        params = best_params,
        train_set = set_tr
    )

    pred = model_tmp.predict(X_va)
    roc_score = roc_auc_score(y_va, pred)
    valid_score.append(roc_score)

    print(f'fold:{fold + 1} roc_score: {roc_score}')

cv_score = np.mean(valid_score)
print(f'cv_score: {cv_score}')

fold:1 roc_score: 0.9349439873660239
fold:2 roc_score: 0.9380032652368597
fold:3 roc_score: 0.9365064702597956
fold:4 roc_score: 0.932261046517993
fold:5 roc_score: 0.9319261065676969
cv_score: 0.9347281751896739


In [12]:
best_model = lgb.train(
    params = best_params,
    train_set = dtrain
)

pd.DataFrame(best_model.feature_importance(), index=X_train.columns )

,0
CreditScore,198
Age,304
Tenure,86
Balance,222
EstimatedSalary,143
is_young_stable,16
is_active_churn,1
is_peak_churn,45
is_senior_retire,0
is_new_customer,12


## 学習済みモデルを用いてテストデータを予測

In [13]:
pred_df = test.drop(['Exited'], axis = 1)
pred_test = best_model.predict(pred_df)

In [14]:
# 1. 欠損値を埋める
test_encoded = test.fillna(0)

# 2. 提出用の「確率」を予測する (predict ではなく predict_proba)
# ※学習時と同じ列にするため、もし 'id' や 'Exited' が残っていれば drop してください
X_test_final = test_encoded.drop(columns=['id', 'Exited'], errors='ignore')
pred_proba = best_model.predict(X_test_final)

In [15]:
np.savetxt('../../model/pred_test_proba_card_tenure.txt', pred_proba)

print("予測確率の保存が完了しました。左のフォルダから 'pred_test_proba_card_tenure.txt' をダウンロードするか、次のファイルで読み込んでください。")

予測確率の保存が完了しました。左のフォルダから 'pred_test_proba_card_tenure.txt' をダウンロードするか、次のファイルで読み込んでください。
